In [ ]:
#AKT
import heapq
import random
import time
from math import floor

# ---------------------------------------------------------------
# 1. TRẠNG THÁI PUZZLE
# ---------------------------------------------------------------
class State:
    def __init__(self, board, n, parent=None, move=None, g=0):
        self.board  = board
        self.n      = n
        self.parent = parent
        self.move   = move
        self.g      = g
        self.h      = self._manhattan()
        self.f      = self.g + self.h
    def _manhattan(self):
        dist = 0
        for idx, val in enumerate(self.board):
            if val != 0:
                goal_r = (val - 1) // self.n
                goal_c = (val - 1) %  self.n
                cur_r  = idx // self.n
                cur_c  = idx %  self.n
                dist  += abs(goal_r - cur_r) + abs(goal_c - cur_c)
        return dist
    def is_goal(self):
        return self.board == list(range(1, self.n * self.n)) + [0]
    def get_neighbors(self):
        neighbors = []
        z = self.board.index(0)
        r, c = z // self.n, z % self.n
        directions = {'UP':(-1,0), 'DOWN':(1,0), 'LEFT':(0,-1), 'RIGHT':(0,1)}
        for mv, (dr, dc) in directions.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n and 0 <= nc < self.n:
                new_board = self.board[:]
                ni = nr * self.n + nc
                new_board[z], new_board[ni] = new_board[ni], new_board[z]
                # cost(S→N) = 1 (mỗi bước di chuyển có chi phí 1)
                neighbors.append(State(new_board, self.n, self, mv, self.g + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

    def __hash__(self):
        return hash(tuple(self.board))

    def __eq__(self, other):
        return self.board == other.board


# ---------------------------------------------------------------
# 2. THUẬT GIẢI AKT
# ---------------------------------------------------------------

def AKT(start_board, n):
    """
    Giải N-puzzle theo đúng thuật giải AKT trong tài liệu.
    Trả về (danh_sách_bước, số_đỉnh_đã_đóng, thời_gian)
    """
    t0 = time.time()

    # --- Bước 1: Mở đỉnh đầu tiên S ---
    S = State(start_board, n)
    if S.is_goal():
        return [], 0, 0.0

    open_dict  = {tuple(S.board): S}
    closed_set = set()
    nodes_closed = 0

    while True:
        if not open_dict:
            return None, nodes_closed, time.time() - t0  # Thất bại

        min_f = min(s.f for s in open_dict.values())

        candidates = [s for s in open_dict.values() if s.f == min_f]
        goal_candidate = next((s for s in candidates if s.is_goal()), None)
        if goal_candidate:
            path = _trace_path(goal_candidate)
            return path, nodes_closed, time.time() - t0
N = candidates[0] if len(candidates) == 1 else random.choice(candidates)
        del open_dict[tuple(N.board)]
        closed_set.add(tuple(N.board))
        nodes_closed += 1

        for S_next in N.get_neighbors():
            key = tuple(S_next.board)
            if key in closed_set:
                continue

            if key not in open_dict or S_next.g < open_dict[key].g:
                open_dict[key] = S_next

def _trace_path(goal_state):
    """Truy vết đường đi từ đỉnh đích về đỉnh gốc."""
    path = []
    node = goal_state
    while node.parent is not None:
        path.append(node.move)
        node = node.parent
    path.reverse()
    return path
def is_solvable(board, n):
    flat = [x for x in board if x != 0]
    inv  = sum(1 for i in range(len(flat))
                 for j in range(i+1, len(flat)) if flat[i] > flat[j])
    if n % 2 == 1:
        return inv % 2 == 0
    else:
        blank_row_from_bottom = n - (board.index(0) // n)
        return (inv % 2 == 1) if (blank_row_from_bottom % 2 == 0) else (inv % 2 == 0)

def generate_puzzle(n, shuffles=40):
    """Sinh puzzle bằng cách trộn ngẫu nhiên từ trạng thái đích."""
    board = list(range(1, n*n)) + [0]
    for _ in range(shuffles):
        z = board.index(0)
        r, c = z // n, z % n
        moves = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n:
                moves.append(nr*n + nc)
        ni = random.choice(moves)
        board[z], board[ni] = board[ni], board[z]
    return board


# ---------------------------------------------------------------
# 5. HIỂN THỊ BẢNG
# ---------------------------------------------------------------

def print_board(board, n, title=""):
    if title:
        print(f"\n{title}")
    w   = len(str(n*n))
    sep = "+" + ("-"*(w+2) + "+") * n
    print(sep)
    for row in range(n):
        cells = []
        for col in range(n):
            v = board[row*n + col]
            s = str(v) if v != 0 else " "
            cells.append(s.rjust(w))
        print("| " + " | ".join(cells) + " |")
        print(sep)


def show_solution_steps(start_board, moves, n, max_show=5):
    """Hiển thị tối đa max_show bước đầu và bước cuối."""
    board = start_board[:]
    print_board(board, n, "🔢 Trạng thái ban đầu")
    total = len(moves)
    for i, mv in enumerate(moves):
        z  = board.index(0)
        r, c = z // n, z % n
        dr, dc = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}[mv]
        ni = (r+dr)*n + (c+dc)
        board[z], board[ni] = board[ni], board[z]
        if i < max_show or i == total - 1:
            label = f"Bước {i+1}/{total}: {mv}"
            if i == max_show and total > max_show + 1:
                print(f"\n  ... (bỏ qua {total - max_show - 1} bước giữa) ...")
print_board(board, n, label)
# ---------------------------------------------------------------
# 6. HÀM CHÍNH
# ---------------------------------------------------------------

def run(n=5, shuffles=15, show_steps=True):
    print(f"\n{'='*52}")
    print(f"  THUẬT GIẢI AKT – {n}×{n} PUZZLE  ({n*n-1}-puzzle)")
    print(f"{'='*52}")

    board = generate_puzzle(n, shuffles)
    assert is_solvable(board, n), "Puzzle không giải được (lỗi sinh puzzle)!"

    print(f"  Kích thước: {n}×{n}   |   Số lần trộn: {shuffles}")
    print_board(board, n, "📌 Trạng thái ban đầu")

    print("\n⏳ Đang chạy thuật giải AKT...")
    moves, closed, elapsed = AKT(board[:], n)

    if moves is None:
        print("❌ Không tìm được lời giải!")
        return

    print(f"\n✅ KẾT QUẢ:")
    print(f"   Số bước giải:        {len(moves)}")
    print(f"   Số đỉnh đã đóng:     {closed:,}")
    print(f"   Thời gian:           {elapsed:.4f} giây")
    print(f"   Trình tự di chuyển:  {' → '.join(moves)}")

    if show_steps:
        show_solution_steps(board, moves, n, max_show=5)

    print(f"\n{'='*52}")
    return moves


# ---------------------------------------------------------------
# 7. CHẠY THỬ NGHIỆM
# ---------------------------------------------------------------

if __name__ == "__main__":

    # --- Ví dụ 1: 5×5 (24-puzzle) – n > 4, trộn ít để giải nhanh ---
    run(n=5, shuffles=12, show_steps=True)

    # --- Ví dụ 2: 6×6 (35-puzzle) – trộn rất ít ---
    run(n=6, shuffles=8, show_steps=False)

    # --- Ví dụ 3: Bảng tùy chỉnh 5×5 ---
    print("\n" + "="*52)
    print("  BẢNG TỰ NHẬP 5×5")
    print("="*52)
    custom = [1,  2,  3,  4,  5,
              6,  7,  8,  9,  10,
              11, 12, 0,  13, 15,
              16, 17, 18, 14, 20,
              21, 22, 23, 19, 24]
    n = 5
    if is_solvable(custom, n):
        print_board(custom, n, "Bảng tùy chỉnh:")
        moves, closed, t = AKT(custom[:], n)
        if moves:
            print(f"✅ Giải được: {len(moves)} bước | {closed} đỉnh đóng | {t:.4f}s")
            print(f"   {' → '.join(moves)}")
    else:
        print("❌ Bảng này không giải được!")

In [ ]:
import heapq

# 1. Định nghĩa cấu trúc đồ thị (danh sách kề)
# format: {node: [(neighbor, weight), ...] }
graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('A', 1), ('D', 2), ('E', 5)],
    'C': [('A', 4), ('F', 1)],
    'D': [('B', 2), ('G', 1)],
    'E': [('B', 5), ('H', 2)],
    'F': [('C', 1), ('I', 3)],
    'G': [('D', 1), ('J', 3)],
    'H': [('E', 2), ('K', 1)],
    'I': [('F', 3), ('L', 2)],
    'J': [('G', 3), ('M', 2)],
    'K': [('H', 1), ('N', 4)],
    'L': [('I', 2), ('Goal', 1)],
    'M': [('J', 2), ('Goal', 4)],
    'N': [('K', 4), ('Goal', 1)],
    'Goal': []
}

# 2. Định nghĩa vị trí (x, y) cho mỗi đỉnh để tính heuristic (khoảng cách Manhattan)
# Đây là một giả định. Nếu đồ thị của bạn không có tọa độ, bạn có thể cần một heuristic khác
positions = {
    'A': (0, 0),
    'B': (1, 0),
    'C': (0, 1),
    'D': (2, 0),
    'E': (1, 1),
    'F': (0, 2),
    'G': (3, 0),
    'H': (2, 1),
    'I': (1, 2),
    'J': (4, 0),
    'K': (3, 1),
    'L': (2, 2),
    'M': (5, 0),
    'N': (4, 1),
    'Goal': (3, 3) # Vị trí mục tiêu
}

# Hàm heuristic (khoảng cách Manhattan)
def heuristic(node, goal_node):
    x1, y1 = positions[node]
    x2, y2 = positions[goal_node]
    return abs(x1 - x2) + abs(y1 - y2)

# 3. Cài đặt thuật toán A*
def a_star_search(graph, start_node, goal_node):
    # Priority queue: (f_score, node)
    open_set = [(0 + heuristic(start_node, goal_node), start_node)]

    # g_score: chi phí từ start_node đến node hiện tại
    g_score = {node: float('inf') for node in graph}
    g_score[start_node] = 0

    # came_from: để tái tạo đường đi
    came_from = {}

    # f_score: g_score + heuristic
    f_score = {node: float('inf') for node in graph}
    f_score[start_node] = heuristic(start_node, goal_node)

    while open_set:
        current_f_score, current_node = heapq.heappop(open_set)

        if current_node == goal_node:
            # Tái tạo đường đi
            path = []
            while current_node in came_from:
                path.append(current_node)
                current_node = came_from[current_node]
            path.append(start_node)
            return path[::-1] # Trả về đường đi đảo ngược

        for neighbor, weight in graph[current_node]:
            tentative_g_score = g_score[current_node] + weight

            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current_node
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = g_score[neighbor] + heuristic(neighbor, goal_node)
                heapq.heappush(open_set, (f_score[neighbor], neighbor))

    return None # Không tìm thấy đường đi

# 4. Ví dụ sử dụng
start_node = 'A'
goal_node = 'Goal'

path = a_star_search(graph, start_node, goal_node)

if path:
    print(f"Đường đi ngắn nhất từ {start_node} đến {goal_node}: {path}")
    total_cost = 0
    for i in range(len(path) - 1):
        u = path[i]
        v = path[i+1]
        for neighbor, weight in graph[u]:
            if neighbor == v:
                total_cost += weight
                break
    print(f"Tổng chi phí của đường đi: {total_cost}")
else:
    print(f"Không tìm thấy đường đi từ {start_node} đến {goal_node}.")

In [11]:
#A*
import heapq
import math
from itertools import combinations

class TSP_AStar:
    def __init__(self, distance_matrix, start=0):
        """
        distance_matrix: ma trận khoảng cách giữa các thành phố
        start: thành phố bắt đầu (mặc định 0)
        """
        self.n = len(distance_matrix)
        self.dist = distance_matrix
        self.start = start

    def heuristic(self, visited, current):
        """
        Ước lượng chi phí từ trạng thái hiện tại đến khi kết thúc:
        - Nếu đã thăm hết thành phố: khoảng cách từ current về start
        - Ngược lại: chi phí cây khung nhỏ nhất của các thành phố chưa thăm + khoảng cách gần nhất từ current đến các thành phố chưa thăm
        """
        unvisited = [city for city in range(self.n) if city not in visited]

        if not unvisited:  # Đã thăm hết
            return self.dist[current][self.start]

        # Tính MST trên tập unvisited + current (dùng Prim)
        nodes = unvisited + [current]
        mst_cost = self.mst_prim(nodes)

        # Heuristic đơn giản: chi phí MST + khoảng cách gần nhất từ current đến unvisited
        min_to_unvisited = min(self.dist[current][city] for city in unvisited)

        return mst_cost + min_to_unvisited

    def mst_prim(self, nodes):
        """Tính chi phí cây khung nhỏ nhất trên tập nodes"""
        if len(nodes) <= 1:
            return 0

        n_nodes = len(nodes)
        node_to_idx = {node: i for i, node in enumerate(nodes)}

        # Ma trận khoảng cách con
        dist_sub = [[0]*n_nodes for _ in range(n_nodes)]
        for i in range(n_nodes):
            for j in range(n_nodes):
                dist_sub[i][j] = self.dist[nodes[i]][nodes[j]]

        # Prim's algorithm
        visited = [False]*n_nodes
        min_edge = [math.inf]*n_nodes
        min_edge[0] = 0
        total_cost = 0

        for _ in range(n_nodes):
            u = -1
            for i in range(n_nodes):
                if not visited[i] and (u == -1 or min_edge[i] < min_edge[u]):
                    u = i

            visited[u] = True
            total_cost += min_edge[u]

            for v in range(n_nodes):
                if not visited[v] and dist_sub[u][v] < min_edge[v]:
                    min_edge[v] = dist_sub[u][v]

        return total_cost

    def solve(self):
        """Giải TSP bằng A*"""

        start_state = (0, 0, (self.start,), self.start)
        pq = []
        heapq.heappush(pq, start_state)

        best_cost = math.inf
        best_path = None
        visited_states = {}

        while pq:
            f, g, visited, current = heapq.heappop(pq)


            state_key = (visited, current)
            if state_key in visited_states and visited_states[state_key] <= g:
                continue
            visited_states[state_key] = g


            if len(visited) == self.n:
total = g + self.dist[current][self.start]
                if total < best_cost:
                    best_cost = total
                    best_path = list(visited) + [self.start]
                continue


            for nxt in range(self.n):
                if nxt in visited:
                    continue

                new_g = g + self.dist[current][nxt]
                if new_g >= best_cost:  # Cắt tỉa
                    continue

                new_visited = visited + (nxt,)
                h = self.heuristic(new_visited, nxt)
                new_f = new_g + h

                heapq.heappush(pq, (new_f, new_g, new_visited, nxt))

        return best_path, best_cost
if __name__ == "__main__":

    dist_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    tsp_solver = TSP_AStar(dist_matrix, start=0)
    path, cost = tsp_solver.solve()

    print("Đường đi tối ưu tìm được:", path)
    print("Tổng chi phí:", cost)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 100)